# 정규표현식 응용: 옵션·범위·측정값 검증

공학프로그래밍입문 · 2026학년도 1학기 · 핵심 활동 약 40~50분

**목표는 문법 암기보다 원리를 이해하고, 코딩 AI에 정확한 질문·지시를 하며, 결과를 검토하는 능력입니다.**
코드는 예측한 뒤 실행하고, 값·조건을 직접 바꾸며 확인합니다. AI 도구를 사용할 수 없으면 제시된 가상 답변을 동료와 검토합니다.

준비: 13주차 search/fullmatch, 예외 처리

## 오늘 설명할 수 있어야 하는 것

- 대소문자·줄바꿈 옵션이 일치 범위를 어떻게 바꾸는지 설명한다.
- 패턴 검증과 숫자 의미 검증을 별도 단계로 구성한다.


## 1. 먼저 원리를 읽기

`IGNORECASE`는 대소문자, `MULTILINE`은 ^와 $의 줄 경계, `DOTALL`은 점(.)의 줄바꿈 포함 여부에 영향을 줍니다.
패턴이 맞는다고 측정값이 유효한 것은 아닙니다. 예를 들어 음수 강수량을 거절하려면 숫자로 바꾼 뒤 범위도 확인해야 합니다. 여기서는 허용 문법 자체를 작게 한정합니다.


## 2. 실행 전 예측하기

`^rain`에 MULTILINE을 넣으면 매칭 개수는 어떻게 달라질까요? DOTALL을 넣으면 `a.b`가 줄바꿈을 포함한 `"a\nb"`와 맞을까요?


아래 기록을 먼저 채운 뒤 다음 코드 셀을 실행하세요. 정답과 다르면 예측에 사용한 가정을 고칩니다.

- 내 예상 결과:
- 그렇게 생각한 이유:
- 실행 뒤 발견한 차이:


In [ ]:
import re

text = "rain=1\nrain=2"
assert len(re.findall(r"^rain", text)) == 1
assert len(re.findall(r"^rain", text, re.MULTILINE)) == 2
assert re.fullmatch(r"a.b", "a\nb") is None
assert re.fullmatch(r"a.b", "a\nb", re.DOTALL) is not None
print("옵션별 매칭 차이를 확인했습니다")


### 관찰을 원리와 연결하기

옵션은 편의 기능이 아니라 허용하는 입력 범위를 바꿉니다. 전부 켜는 대신 필요한 옵션과 그 효과를 반례로 확인하세요.


다음 셀의 입력이나 조건을 하나 바꾸세요. 바꾼 값, 예상한 변화, 실제 결과를 적습니다. 단순 재실행만으로 이 활동을 마치지 않습니다.


In [ ]:
# 이 예제에서만 정한 문법: rain=비음수 정수 또는 소수+mm
measurement_pattern = re.compile(r"rain=([0-9]+(?:\.[0-9]+)?)mm", re.IGNORECASE)
match = measurement_pattern.fullmatch("RAIN=12.5mm")
assert match is not None
print("추출 문자열:", match.group(1), "숫자:", float(match.group(1)))
assert float(match.group(1)) == 12.5
assert measurement_pattern.fullmatch("rain=-1mm") is None
assert measurement_pattern.fullmatch("rain=1mm trailing") is None


## 3. 작은 기능을 직접 완성하기

`parse_measurement(text)`는 `rain=12.5mm`처럼 형식이 정확한 문자열에서 숫자를 `float`로 반환합니다.
rain과 mm는 대소문자를 무시합니다. 숫자는 ASCII 정수 또는 소수(점 양쪽 한 자리 이상), 범위는 **0~500 mm 양 끝 포함**입니다. 공백·부호·지수 표기·후행 문자는 허용하지 않습니다. 문법 또는 범위 위반은 `ValueError`입니다. 이 상한은 실습용 계약입니다.


먼저 입력·출력 예를 하나 더 직접 만드세요. 모르는 문법은 도움말이나 AI에 질문해도 됩니다. AI 코드 전체를 받기 전에 필요한 처리 단계를 자신의 말로 2~3줄 적으세요.

- 내가 정한 추가 입력과 예상 출력:
- 처리 단계:


In [ ]:
def parse_measurement(text):
    # TODO: 전체 매칭 -> 숫자 변환 -> 실습에서 정한 범위 확인
    raise NotImplementedError("문자열 형식과 숫자 의미를 분리해 확인하세요")


### 공개 점검

아래 점검 셀을 실행합니다. **“미완료”는 통과가 아닙니다.** 위 함수를 구현한 뒤 다시 실행하세요.
실패하면 기대값을 바꾸기보다 입력·계약·구현 중 무엇이 다른지 조사합니다. 공개 시험만 통과했다고 모든 입력에 옳다는 뜻은 아닙니다.


In [ ]:
def check_answer(candidate):
    for text, expected in [("rain=0mm", 0.), ("RAIN=12.5MM", 12.5), ("rain=500mm", 500.)]:
        result = candidate(text)
        assert isinstance(result, float) and result == expected
    for text in ["rain=-1mm", "rain=501mm", "rain=1e2mm", "rain=.5mm", " rain=1mm", "rain=1mm\n", "rain=1mm extra"]:
        try:
            candidate(text)
        except ValueError:
            pass
        else:
            raise AssertionError(f"거절해야 하는 입력: {text!r}")
    print("공개 점검 통과: 옵션·전체 범위·문법·경계값")

try:
    check_answer(parse_measurement)
except NotImplementedError:
    print("미완료: 위 TODO를 구현한 뒤 이 셀을 다시 실행하세요. 아직 점검을 통과하지 않았습니다.")


## 4. AI 제안을 검토하고 수정 지시하기

다음은 오류를 검토하기 위해 만든 **가상 AI 답변**입니다. 실제 AI 응답이나 정답이 아닙니다.
실행 전에 어떤 요구사항을 놓쳤는지 찾고, 반례가 그 문제를 어떻게 드러내는지 설명하세요.


In [ ]:
def ai_suggestion(text):
    match = re.search(r"rain=([0-9.]+)mm", text, re.IGNORECASE)
    return float(match.group(1))

print("후행 문자 허용:", ai_suggestion("rain=1mm extra"))
print("실습 상한 초과:", ai_suggestion("rain=999mm"))
assert ai_suggestion("rain=999mm") == 999


### 검토 질문

검색 결과를 숫자로 바꾸는 것만으로 형식·범위 계약을 지킬 수 없습니다. 또한 `[0-9.]+`는 여러 점도 허용합니다. 규칙을 세 부분(문법·전체 범위·값 범위)으로 나눠 수정 요청을 쓰세요.


- 위 제안이 어긴 요구사항:
- 오류를 드러내는 입력과 기대 결과:
- 한 문장 수정 지시:


### 코딩 AI에게 보낼 요청 작성

허용할 문자열 예와 문법, IGNORECASE만 필요한 이유, 전체 매칭, 반환 단위 mm, 0~500 범위, ValueError 정책을 명시한다. 0·500 경계와 501·후행 문자·지수 표기 반례를 시험하도록 요청한다.


위 문장을 그대로 복사하는 대신 오늘의 실제 입력·기대값·실패 조건을 넣어 요청문을 완성하세요.
아래 틀로 **구현 전 계획**을 먼저 요청하고, 받은 코드에는 새 반례를 하나 추가합니다.

```text
목적과 환경: Python / Google Colab에서 ... 하려고 한다.
입력: 자료형·구조·단위는 ... 이며, 예시는 ... 이다.
출력: ... 형식으로 ... 를 반환해야 한다.
조건: ... 는 포함하고 ... 는 제외한다. 잘못된 입력은 ... 로 처리한다.
검증: 정상 예 ... / 경계 예 ... / 반례 ... 의 기대 결과는 ... 이다.
먼저 모호한 조건을 짚고 처리 계획을 설명해 줘. 그 뒤 짧은 코드와 검증을 제시해 줘.
```

- 내가 보낸 최초 요청:
- 검토 후 보낸 수정 요청:
- AI 제안에서 채택·수정·거절한 부분과 근거:
- 내가 추가한 시험의 입력·기대값·실제값:


## 5. 새 상황에 옮겨 보기

실무에서 상한 500을 임의로 쓰면 어떤 문제가 생길까요? 관측 장비·시간 간격·품질관리 기준 중 무엇을 먼저 확인해야 할까요?


이 활동은 한 가지 구현을 암기했는지가 아니라 요구사항이 달라졌을 때 무엇을 다시 질문하는지 확인합니다.


## 마무리 기록

- 오늘 사용한 핵심 원리를 자신의 말로 한 문장:
- 결과를 신뢰할 근거(시험 입력·기대값·실제값):
- 아직 확인하지 않은 조건 또는 적용 한계:
- AI 없이 설명할 수 있는 코드 한 줄과 그 역할:

**완료 기준:** 예측을 남겼고, 입력·조건을 바꿨고, TODO를 구현해 공개 점검을 통과했고, 새 반례와 수정 지시를 작성했으며, 결과의 한계를 설명할 수 있습니다.

[과목의 실습 방식과 점검 기준](../LEARNING_GUIDE.md)을 참고하세요.


## 출처와 개편 범위

    기존 강의 노트북의 주제를 바탕으로 핵심 예제, 합성 연습 데이터, AI 검토 활동을 새로 구성했습니다.
    원본 파일명은 유지했고, 원본 Drive 링크와 개편 전·후 해시는 강의 폴더의 `materials-manifest.json`에 기록합니다.
    노트북 실행에 외부 AI API나 원본 로컬 ZIP은 필요하지 않습니다. 학습용 오류 제안은 반례를 확인하는 코드이며, 미완성 TODO와 구분합니다.
